# Regressão Linear Simples — Exemplo 02

Prever o **valor de venda real** do item com base no **valor de custo** do produto.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Conexão com o Banco de Dados PostgreSQL

In [ ]:
usuario = "datadt_data_analytics"
senha = "DataAnalytics$100"
host = "postgresql-datadt.alwaysdata.net"
porta = "5432"
banco = "datadt_digital_corporativo"

engine = create_engine(
    f"postgresql+psycopg2://{usuario}:{senha}@{host}:{porta}/{banco}"
)

## 2. Consulta SQL

Cada linha representa um item vendido em uma nota fiscal.
- `X` = `valor_custo` do produto
- `Y` = `valor_venda_real` praticado na venda

In [ ]:
sql = """
SELECT 
    inf.id AS id_item,
    p.id AS id_produto,
    p.nome AS produto,
    p.valor_custo,
    p.valor_venda AS valor_venda_cadastrado,
    inf.valor_unitario as valor_venda_real
FROM vendas.item_nota_fiscal inf
JOIN vendas.produto p 
    ON p.id = inf.id_produto
WHERE p.valor_custo IS NOT NULL
  AND inf.valor_unitario IS NOT NULL
  AND p.valor_custo > 0
  AND inf.valor_unitario > 0
ORDER BY inf.id;
"""

## 3. Carregando os Dados

In [ ]:
df = pd.read_sql(sql, engine)

print("Primeiras linhas da base:")
print(df.head())

print("\nInformações da base:")
print(df.info())

print("\nResumo estatístico:")
print(df[["valor_custo", "valor_venda_real"]].describe())

## 4. Tratamento Básico dos Dados

In [ ]:
df = df.dropna(subset=["valor_custo", "valor_venda_real"])

df = df[
    (df["valor_custo"] > 0) &
    (df["valor_venda_real"] > 0)
]

print("Quantidade de registros após tratamento:")
print(len(df))

## 5. Definindo X e Y

In [ ]:
X = df[["valor_custo"]]
y = df["valor_venda_real"]

## 6. Divisão em Treino e Teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=42
)

## 7. Treinando o Modelo

Usando `fit_intercept=False` para forçar a reta a passar pela origem.

In [ ]:
modelo = LinearRegression(fit_intercept=False)
modelo.fit(X_train, y_train)

## 8. Coeficientes do Modelo

In [ ]:
intercepto = modelo.intercept_
coeficiente = modelo.coef_[0]

print("Modelo treinado:")
print(f"Intercepto: {intercepto:.2f}")
print(f"Coeficiente: {coeficiente:.2f}")

print("\nEquação da regressão:")
print(f"valor_venda_real = {intercepto:.2f} + {coeficiente:.2f} * valor_custo")

## 9. Fazendo Previsões

In [ ]:
y_pred = modelo.predict(X_test)

resultado = pd.DataFrame({
    "valor_custo": X_test["valor_custo"],
    "valor_real": y_test,
    "valor_previsto": y_pred
})

print("Comparação entre valor real e valor previsto:")
print(resultado.head(10))

## 10. Avaliação do Modelo

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Métricas de avaliação:")
print(f"MAE  - Erro médio absoluto: {mae:.2f}")
print(f"MSE  - Erro quadrático médio: {mse:.2f}")
print(f"R²   - Coeficiente de determinação: {r2:.4f}")

## 11. Visualização dos Dados e da Reta de Regressão

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(X, y, label="Dados reais")
plt.plot(X, modelo.predict(X), label="Reta de regressão")

plt.title("Regressão Linear Simples")
plt.xlabel("Valor de custo do produto")
plt.ylabel("Valor de venda real")
plt.legend()
plt.grid(True)

arquivo_grafico = "grafico_regressao_exemplo02.png"
plt.savefig(arquivo_grafico, dpi=300, bbox_inches="tight")
plt.show()

print(f"\nGráfico salvo em: {arquivo_grafico}")

## 12. Simulação de Previsão

Prever o valor de venda real de um produto cujo custo é R$ 100,00.

In [ ]:
novo_custo = pd.DataFrame({
    "valor_custo": [100]
})

valor_estimado = modelo.predict(novo_custo)

print("Simulação:")
print(f"Para um produto com custo de R$ 100,00, o valor de venda estimado é R$ {valor_estimado[0]:.2f}")